# Sign language recognition — ViT-B/16 (torchvision `IMAGENET1K_V1`)

- Full 224×224 frames (no crop, no resize) → ViT-B/16 per frame → temporal Transformer → 100 classes.
- **Validation**: 5-fold, grouped by original video id (`xxx_1.mp4`, `xxx_2.mp4`… are colour-jittered copies of `xxx.mp4` and always stay in the same fold), exactly stratified so every fold's val contains all 100 classes. Metric = Macro-F1 on **original** val videos, out-of-fold.
- **Part A**: frozen ViT, features of all 30 frames cached once → 16 frames sampled per clip (random per segment when training) → fast experiments (loss comparison, logit adjustment).
- **Part B** (optional, heavy): fine-tune the last ViT blocks end-to-end, head warm-started from Part A.
- **Submission**: `cv_submission.zip` → `cv_submission.csv` (private test).

Set `SMOKE=1` in the environment for a quick end-to-end check on a few videos.

In [ ]:
import os, re, time, math, pickle, random, zipfile, unicodedata
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
SMOKE = bool(int(os.environ.get('SMOKE', '0')))  # quick end-to-end check on a few videos

DATA = 'dataset'
WORK = 'work'
FEAT_DIR = f'{WORK}/feats'
os.makedirs(FEAT_DIR, exist_ok=True)

NUM_CLASSES = 100
N_FOLDS = 5
T = 16          # frames fed to the temporal model
T_MAX = 30      # frames cached per video (videos have <= 30 frames)
SEED = 42
WEIGHTS = ViT_B_16_Weights.IMAGENET1K_V1

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all()
print(f'device={DEVICE} | smoke={SMOKE}')

## 1. Data & grouped folds

In [ ]:
nfc = lambda s: unicodedata.normalize('NFC', s)
with open(f'{DATA}/label_mapping.pkl', 'rb') as f:
    raw_mapping = pickle.load(f)
label_to_idx = {nfc(k): v for k, v in raw_mapping.items()}
idx_to_label = {v: k for k, v in raw_mapping.items()}   # keep the organiser's exact strings for the CSV

rows = []
for label in sorted(os.listdir(f'{DATA}/train')):
    d = f'{DATA}/train/{label}'
    if not os.path.isdir(d):
        continue
    for fn in sorted(os.listdir(d)):
        if fn.endswith('.mp4'):
            stem = fn[:-4]
            rows.append({'path': f'{d}/{fn}', 'label_idx': label_to_idx[nfc(label)],
                         'group': re.sub(r'_\d+$', '', stem), 'is_orig': re.search(r'_\d+$', stem) is None})
df = pd.DataFrame(rows)

# Each group has a single label, so stratify exactly: per class, shuffle groups and deal them round-robin.
# The starting fold rotates per class so fold sizes stay balanced.
rng = np.random.default_rng(SEED)
group_fold, offset = {}, 0
for _, g in df.groupby('label_idx'):
    groups = g.group.unique()
    rng.shuffle(groups)
    for i, grp in enumerate(groups):
        group_fold[grp] = (i + offset) % N_FOLDS
    offset += len(groups)
df['fold'] = df.group.map(group_fold)

if SMOKE:
    df = df[df.group.isin(df.groupby('label_idx').group.unique().apply(lambda g: g[:N_FOLDS]).explode())].reset_index(drop=True)
else:
    df.to_csv(f'{WORK}/folds.csv', index=False)

test_files = {s: sorted(f for f in os.listdir(f'{DATA}/{s}') if f.endswith('.mp4')) for s in ['public_test', 'private_test']}
if SMOKE:
    test_files = {s: v[:20] for s, v in test_files.items()}

orig_counts = np.bincount(df[df.is_orig].label_idx, minlength=NUM_CLASSES)
file_counts = np.bincount(df.label_idx, minlength=NUM_CLASSES)
print(f'train files={len(df)} originals={df.is_orig.sum()} | public={len(test_files["public_test"])} private={len(test_files["private_test"])}')
print(f'originals per class: min={orig_counts.min()} max={orig_counts.max()}')
print(df[df.is_orig].groupby('fold').label_idx.nunique().rename('classes in val'))

## 2. Video reading & frame sampling

In [ ]:
def read_video(path, size=224):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        if f.shape[:2] != (size, size):
            f = cv2.resize(f, (size, size), interpolation=cv2.INTER_AREA)  # whole frame, never cropped
        frames.append(f)
    cap.release()
    if not frames:
        raise ValueError(f'Could not read any frames from {path}')
    return np.stack(frames)  # (N, H, W, 3) uint8


def sample_idx(n, t=T, phase=0.5, train=False):
    """Split [0, n) into t equal segments and take one frame per segment:
    a random position when training (temporal jitter), else the given phase (0.5 = centre)."""
    edges = np.linspace(0, n, t + 1)
    pos = np.random.rand(t) if train else np.full(t, phase)
    idx = edges[:-1] + pos * (edges[1:] - edges[:-1])
    return np.clip(idx.astype(int), 0, n - 1)


MEAN = torch.tensor(WEIGHTS.transforms().mean).view(1, 3, 1, 1)
STD = torch.tensor(WEIGHTS.transforms().std).view(1, 3, 1, 1)

def to_tensor(frames):
    """uint8 (N,H,W,3) -> normalised float (N,3,H,W). No resize/centre-crop: frames are already 224x224."""
    x = torch.from_numpy(frames).permute(0, 3, 1, 2).float().div_(255)
    return (x - MEAN) / STD

## 3. ViT-B/16 frame features (frozen) — cached once

Per frame: `[CLS token, mean of patch tokens]` after the final LayerNorm → 1536-d. All 30 frames are cached (float16), so any frame-sampling strategy can be tried later for free.

In [ ]:
def vit_frame_features(vit, x):
    """x: (B,3,224,224) normalised -> (B,1536) = [CLS, mean patch token] after the encoder's final LayerNorm."""
    x = vit._process_input(x)
    x = torch.cat([vit.class_token.expand(x.shape[0], -1, -1), x], dim=1)
    x = vit.encoder(x)
    return torch.cat([x[:, 0], x[:, 1:].mean(1)], dim=1)


@torch.no_grad()
def extract_features(paths, out_file, videos_per_batch=4):
    if os.path.exists(out_file):
        z = np.load(out_file)
        print(f'loaded cache {out_file}')
        return z['feats'], z['lens']
    vit = vit_b_16(weights=WEIGHTS).eval().to(DEVICE)
    half = DEVICE != 'cpu'
    if half:
        vit = vit.half()
    feats = np.zeros((len(paths), T_MAX, 1536), dtype=np.float16)
    lens = np.zeros(len(paths), dtype=np.int16)
    for s in tqdm(range(0, len(paths), videos_per_batch), desc=os.path.basename(out_file)):
        clips = [read_video(p)[:T_MAX] for p in paths[s:s + videos_per_batch]]
        x = to_tensor(np.concatenate(clips)).to(DEVICE)
        f = vit_frame_features(vit, x.half() if half else x).float().cpu().numpy()
        o = 0
        for j, c in enumerate(clips):
            n = len(c)
            feats[s + j, :n] = f[o:o + n]
            feats[s + j, n:] = f[o + n - 1]   # pad short clips with their last frame
            lens[s + j] = n
            o += n
    np.savez(out_file, feats=feats, lens=lens)
    del vit
    return feats, lens


tag = '_smoke' if SMOKE else ''
t0 = time.time()
train_feats, train_lens = extract_features(df.path.tolist(), f'{FEAT_DIR}/vitb16_train{tag}.npz')
test_feats = {s: extract_features([f'{DATA}/{s}/{f}' for f in fs], f'{FEAT_DIR}/vitb16_{s}{tag}.npz')
              for s, fs in test_files.items()}
print(f'features ready in {time.time() - t0:.0f}s | train {train_feats.shape}')

## 4. Temporal head, losses

In [ ]:
class TemporalHead(nn.Module):
    """Frame features (B,T,1536) -> CLS-token Transformer over time -> logits."""
    def __init__(self, in_dim=1536, d=384, depth=2, heads=6, drop=0.2, n_cls=NUM_CLASSES, t_max=T_MAX):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(drop), nn.Linear(in_dim, d))
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.zeros(1, t_max + 1, d))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, activation='gelu', batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.fc = nn.Linear(d, n_cls)

    def forward(self, x):
        x = self.inp(x)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], dim=1) + self.pos[:, :x.shape[1] + 1]
        return self.fc(self.norm(self.enc(x)[:, 0]))


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma, self.weight, self.ls = gamma, weight, label_smoothing

    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, weight=self.weight, label_smoothing=self.ls, reduction='none')
        pt = F.softmax(logits, 1).gather(1, y[:, None]).squeeze(1)
        return ((1 - pt) ** self.gamma * ce).mean()


LABEL_SMOOTH = 0.1
# Class weights from *original* video counts (the _k copies don't change how rare a sign really is);
# power 0.5 softens the ~10x imbalance.
CLASS_W = torch.tensor((orig_counts.mean() / np.maximum(orig_counts, 1)) ** 0.5, dtype=torch.float32)

def make_criterion(kind):
    if kind == 'ce':
        return nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    if kind == 'weighted':
        return nn.CrossEntropyLoss(weight=CLASS_W.to(DEVICE), label_smoothing=LABEL_SMOOTH)
    if kind == 'focal':
        return FocalLoss(gamma=2.0)
    raise ValueError(kind)

## 5. Part A — train on cached features (5-fold CV)

In [ ]:
class FeatDataset(Dataset):
    def __init__(self, feats, lens, labels, train):
        self.feats, self.lens, self.labels, self.train = feats, lens, labels, train

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, i):
        idx = sample_idx(int(self.lens[i]), T, train=self.train)
        return torch.from_numpy(self.feats[i, idx].astype(np.float32)), int(self.labels[i])


EPOCHS = 2 if SMOKE else 60
BATCH = 64
LR = 5e-4

def cosine_with_warmup(opt, total, warmup):
    return torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup))))


@torch.no_grad()
def predict_feats(model, feats, lens, phases=(0.5,)):
    """Mean logits over temporal TTA phases."""
    model.eval()
    out = []
    for s in range(0, len(feats), 256):
        logits = 0
        for ph in phases:
            x = np.stack([feats[i, sample_idx(int(lens[i]), T, phase=ph)] for i in range(s, min(s + 256, len(feats)))])
            logits = logits + model(torch.from_numpy(x.astype(np.float32)).to(DEVICE))
        out.append((logits / len(phases)).cpu())
    return torch.cat(out).numpy()


def macro_f1(y, logits):
    return f1_score(y, logits.argmax(1), average='macro', labels=np.arange(NUM_CLASSES), zero_division=0)


def train_fold(fold, loss_kind, verbose=True):
    seed_all(SEED + fold)
    tr = np.where(df.fold != fold)[0]
    va = np.where((df.fold == fold) & df.is_orig)[0]
    y = df.label_idx.values
    loader = DataLoader(FeatDataset(train_feats[tr], train_lens[tr], y[tr], train=True),
                        batch_size=BATCH, shuffle=True, drop_last=True)
    model = TemporalHead().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    sched = cosine_with_warmup(opt, EPOCHS * len(loader), 3 * len(loader))
    crit = make_criterion(loss_kind)
    for ep in range(EPOCHS):
        model.train()
        tot = 0
        for x, t in loader:
            x, t = x.to(DEVICE), t.to(DEVICE)
            loss = crit(model(x), t)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        if verbose and ((ep + 1) % 10 == 0 or ep == EPOCHS - 1):
            f1 = macro_f1(y[va], predict_feats(model, train_feats[va], train_lens[va]))
            print(f'  fold {fold} ep {ep + 1:3d} loss {tot / len(loader):.4f} val macro-F1 {f1 * 100:.2f}')
    # No best-epoch picking on val: last-epoch weights keep the OOF score honest.
    return model, va


def run_cv(loss_kind):
    oof = np.zeros((len(df), NUM_CLASSES), dtype=np.float32)
    models = []
    for fold in range(N_FOLDS):
        model, va = train_fold(fold, loss_kind)
        oof[va] = predict_feats(model, train_feats[va], train_lens[va])
        models.append(model)
    m = df.is_orig.values
    print(f'[{loss_kind}] OOF macro-F1 (originals): {macro_f1(df.label_idx.values[m], oof[m]) * 100:.2f}')
    return models, oof

In [ ]:
LOSSES = ['ce', 'weighted', 'focal']
results = {}
for kind in LOSSES:
    t0 = time.time()
    results[kind] = run_cv(kind)
    print(f'   ({time.time() - t0:.0f}s)')

### Logit adjustment (for Macro-F1)
`logits − τ·log(prior)` removes the bias toward frequent classes. τ is tuned on OOF predictions.

In [ ]:
prior = file_counts / file_counts.sum()     # the model was trained on all files, copies included
LOG_PRIOR = np.log(np.maximum(prior, 1e-12)).astype(np.float32)
m = df.is_orig.values
y_orig = df.label_idx.values[m]

best = (None, None, -1)
print('loss      ' + ' '.join(f'τ={t:<4}' for t in np.arange(0, 1.51, 0.25)))
for kind, (_, oof) in results.items():
    scores = []
    for tau in np.arange(0, 1.51, 0.25):
        f1 = macro_f1(y_orig, oof[m] - tau * LOG_PRIOR)
        scores.append(f1)
        if f1 > best[2]:
            best = (kind, tau, f1)
    print(f'{kind:9s} ' + ' '.join(f'{s * 100:6.2f}' for s in scores))
BEST_LOSS, BEST_TAU, BEST_F1 = best
print(f'\nbest: loss={BEST_LOSS} τ={BEST_TAU} OOF macro-F1={BEST_F1 * 100:.2f}')

## 6. Predict test with the 5 fold models → submission

In [ ]:
TTA_PHASES = (0.25, 0.5, 0.75)

def predict_test(models, feats, lens, tau):
    probs = np.mean([F.softmax(torch.from_numpy(predict_feats(mdl, feats, lens, TTA_PHASES)), 1).numpy() for mdl in models], 0)
    return np.log(probs + 1e-9) - tau * LOG_PRIOR


def write_submission(split, logits, csv_name):
    preds = logits.argmax(1)
    sub = pd.DataFrame({'video_name': test_files[split], 'label': [idx_to_label[int(p)] for p in preds]})
    sub.to_csv(csv_name, index=False, encoding='utf-8')
    return sub


fold_models = results[BEST_LOSS][0]
test_logits = {s: predict_test(fold_models, *test_feats[s], BEST_TAU) for s in test_files}
for s in test_files:
    np.save(f'{WORK}/logits_vitb16_{s}{tag}.npy', test_logits[s])   # kept for ensembling later

write_submission('public_test', test_logits['public_test'], f'{WORK}/public_submission_vitb16{tag}.csv')
sub = write_submission('private_test', test_logits['private_test'], f'cv_submission{tag}.csv')
with zipfile.ZipFile(f'cv_submission{tag}.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(f'cv_submission{tag}.csv', arcname='cv_submission.csv')   # csv at the zip root, as required

print(sub.head())
print(f'{len(sub)} rows | distinct labels predicted: {sub.label.nunique()} | top: {sub.label.value_counts().head(3).to_dict()}')

## 7. Part B (optional) — fine-tune the last ViT blocks end-to-end

Heavy on an M4 (decoding + ViT forward of every frame each epoch). Runs one fold so it can be compared with Part A's score on the same fold. The first `12 − FT_BLOCKS` blocks run under `no_grad`, so memory stays small. The head starts from Part A's head for the same fold (same feature definition).

Augmentation never cuts the frame edges: zoom-out/translate with padding plus brightness/contrast, identical for all frames of a clip.

In [ ]:
RUN_FINETUNE = False
FT_FOLD = 0
FT_BLOCKS = 4
FT_EPOCHS = 2 if SMOKE else 10
FT_BATCH = 4          # videos per step (x T frames)
FT_LR_BACKBONE, FT_LR_HEAD = 2e-5, 2e-4


def augment_clip(frames):
    """Same random zoom-out (0.85-1.0) + shift + colour change for every frame; borders padded, nothing cropped."""
    h, w = frames.shape[1:3]
    s = np.random.uniform(0.85, 1.0)
    tx = np.random.uniform(-1, 1) * (1 - s) * w / 2
    ty = np.random.uniform(-1, 1) * (1 - s) * h / 2
    M = np.float32([[s, 0, (1 - s) * w / 2 + tx], [0, s, (1 - s) * h / 2 + ty]])
    fill = tuple(int(v) for v in frames[0].reshape(-1, 3).mean(0))
    out = np.stack([cv2.warpAffine(f, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=fill) for f in frames])
    alpha, beta = np.random.uniform(0.8, 1.2), np.random.uniform(-20, 20)
    return np.clip(out.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


class VideoDataset(Dataset):
    def __init__(self, paths, labels, train):
        self.paths, self.labels, self.train = paths, labels, train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        frames = read_video(self.paths[i])
        frames = frames[sample_idx(len(frames), T, train=self.train)]
        if self.train:
            frames = augment_clip(frames)
        return to_tensor(frames), int(self.labels[i])


class ViTVideo(nn.Module):
    def __init__(self, head, ft_blocks=FT_BLOCKS):
        super().__init__()
        self.vit = vit_b_16(weights=WEIGHTS)
        self.vit.heads = nn.Identity()
        self.k = len(self.vit.encoder.layers) - ft_blocks
        for p in self.vit.parameters():
            p.requires_grad = False
        for blk in self.vit.encoder.layers[self.k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in self.vit.encoder.ln.parameters():
            p.requires_grad = True
        self.head = head

    def forward(self, x):  # (B,T,3,H,W)
        B, Tn = x.shape[:2]
        v = self.vit
        with torch.no_grad():
            z = v._process_input(x.flatten(0, 1))
            z = torch.cat([v.class_token.expand(z.shape[0], -1, -1), z], 1) + v.encoder.pos_embedding
            for blk in v.encoder.layers[:self.k]:
                z = blk(z)
        for blk in v.encoder.layers[self.k:]:
            z = blk(z)
        z = v.encoder.ln(z)
        f = torch.cat([z[:, 0], z[:, 1:].mean(1)], 1)
        return self.head(f.view(B, Tn, -1))


if RUN_FINETUNE:
    seed_all(SEED)
    head = TemporalHead()
    head.load_state_dict(results[BEST_LOSS][0][FT_FOLD].state_dict())
    model = ViTVideo(head).to(DEVICE)
    tr = np.where(df.fold != FT_FOLD)[0]
    va = np.where((df.fold == FT_FOLD) & df.is_orig)[0]
    y = df.label_idx.values
    tr_loader = DataLoader(VideoDataset(df.path.values[tr], y[tr], True), batch_size=FT_BATCH, shuffle=True, drop_last=True)
    va_loader = DataLoader(VideoDataset(df.path.values[va], y[va], False), batch_size=FT_BATCH)
    opt = torch.optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('vit.')], 'lr': FT_LR_BACKBONE},
        {'params': model.head.parameters(), 'lr': FT_LR_HEAD}], weight_decay=0.05)
    sched = cosine_with_warmup(opt, FT_EPOCHS * len(tr_loader), len(tr_loader))
    crit = make_criterion(BEST_LOSS)
    for ep in range(FT_EPOCHS):
        model.train()
        tot = 0
        for x, t in tqdm(tr_loader, desc=f'ft ep {ep + 1}'):
            x, t = x.to(DEVICE), t.to(DEVICE)
            loss = crit(model(x), t)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        model.eval()
        with torch.no_grad():
            logits = np.concatenate([model(x.to(DEVICE)).cpu().numpy() for x, _ in va_loader])
        print(f'ep {ep + 1} loss {tot / len(tr_loader):.4f} | fold {FT_FOLD} val macro-F1 {macro_f1(y[va], logits) * 100:.2f} '
              f'(+τ={BEST_TAU}: {macro_f1(y[va], logits - BEST_TAU * LOG_PRIOR) * 100:.2f})')
        torch.save(model.state_dict(), f'{WORK}/vitb16_ft_fold{FT_FOLD}.pth')
    part_a = macro_f1(y[va], predict_feats(results[BEST_LOSS][0][FT_FOLD], train_feats[va], train_lens[va]))
    print(f'Part A on the same fold: {part_a * 100:.2f}')